# Factor model sweep with MLflow

Runs every config in `configs/` over one shared price panel and logs each run to
MLflow. The sweep table also reports the deflated Sharpe ratio, which discounts the
best result for the number of configurations tried.


In [ ]:
%pip install -e .. yfinance lightgbm mlflow


In [ ]:
from pathlib import Path

from ml_trading.config import ExperimentConfig
from ml_trading.data import load_prices
from ml_trading.metrics import deflated_sharpe_ratio
from ml_trading.pipeline import sweep
from ml_trading.tracking import log_experiment

configs = [ExperimentConfig.from_yaml(p) for p in sorted(Path("../configs").glob("*.yaml"))]
prices = load_prices(configs[0].data)
table, results = sweep(configs, prices)


In [ ]:
table["deflated_sharpe"] = [
    deflated_sharpe_ratio(
        row["strategy_sharpe"],
        n_trials=len(configs),
        n_days=int(row["strategy_n_days"]),
        skew=row["strategy_skew"],
        kurtosis=row["strategy_kurtosis"] + 3.0,
    )
    for _, row in table.iterrows()
]
display(
    table[
        [
            "name",
            "strategy_sharpe",
            "strategy_sharpe_pvalue",
            "deflated_sharpe",
            "mean_fold_ic",
            "benchmark_sharpe",
        ]
    ]
)


In [ ]:
for result in results:
    log_experiment(result, experiment_name="/Shared/ml_trading_sweep")
